# Daily Challenge – US Superstore Visualization
**Dataset :** US Superstore Data  
**Objectif :** Comparaison Matplotlib vs Seaborn, visualisations interactives, analyse des tendances  
**Outils :** Pandas, Matplotlib, Seaborn, ipywidgets

## Partie 1 – Chargement et nettoyage des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
# Le fichier dataset est dans le dossier ExerciceXP (même fichier partagé)
df = pd.read_excel('../ExerciceXP/US Superstore data.xls')

print('Dimensions :', df.shape)
print('Colonnes   :', df.columns.tolist())
df.head()

In [ ]:
# Vérification des valeurs manquantes et doublons
print('Valeurs manquantes :')
print(df.isnull().sum())
print(f'\nDoublons : {df.duplicated().sum()}')
df = df.drop_duplicates()

# Conversion des dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Feature engineering
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')
df['Profit Margin']    = (df['Profit'] / df['Sales'] * 100).round(2)

print('\nPréparation terminée.')
df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].describe()

## Partie 2 – Visualisations avec Matplotlib

### 2.1 Tendance des ventes dans le temps (interactif)

In [ ]:
# Ventes annuelles par catégorie
yearly_sales = df.groupby(['Order Year', 'Category'])['Sales'].sum().reset_index()
total_yearly = df.groupby('Order Year')['Sales'].sum().reset_index()

def plot_sales_trend(view='All Categories'):
    plt.figure(figsize=(12, 6))

    if view == 'All Categories':
        plt.plot(total_yearly['Order Year'], total_yearly['Sales'],
                 marker='o', linewidth=2.5, color='steelblue', markersize=8)
        # Annoter chaque point avec la valeur
        for _, row in total_yearly.iterrows():
            plt.text(row['Order Year'], row['Sales'] + 5000,
                     f'${row["Sales"]:,.0f}', ha='center', fontsize=9)
        plt.title('Tendance Annuelle des Ventes – Toutes Catégories', fontsize=13, fontweight='bold')
    else:
        cat_data = yearly_sales[yearly_sales['Category'] == view]
        plt.plot(cat_data['Order Year'], cat_data['Sales'],
                 marker='o', linewidth=2.5, color='coral', markersize=8)
        for _, row in cat_data.iterrows():
            plt.text(row['Order Year'], row['Sales'] + 2000,
                     f'${row["Sales"]:,.0f}', ha='center', fontsize=9)
        plt.title(f'Tendance Annuelle des Ventes – {view}', fontsize=13, fontweight='bold')

    plt.xlabel('Année')
    plt.ylabel('Ventes ($)')
    plt.xticks(total_yearly['Order Year'])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

options = ['All Categories'] + list(df['Category'].unique())
dropdown = Dropdown(options=options, value='All Categories', description='Catégorie :')
interact(plot_sales_trend, view=dropdown);

### 2.2 Tendance mensuelle des ventes

In [ ]:
monthly_total = df.groupby('Order Month-Year')['Sales'].sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Ventes mensuelles
axes[0].plot(monthly_total.index.to_timestamp(), monthly_total.values,
             color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_total.index.to_timestamp(), monthly_total.values,
                     alpha=0.2, color='steelblue')
axes[0].set_title('Ventes Mensuelles (toutes années)', fontweight='bold')
axes[0].set_ylabel('Ventes ($)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(True, alpha=0.3)

# Ventes et profit par année (barres côte à côte)
yearly_both = df.groupby('Order Year')[['Sales', 'Profit']].sum().reset_index()
x = np.arange(len(yearly_both))
width = 0.35
axes[1].bar(x - width/2, yearly_both['Sales'],  width, label='Ventes',  color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, yearly_both['Profit'], width, label='Profit',  color='coral',     alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(yearly_both['Order Year'])
axes[1].set_title('Ventes et Profit par Année', fontweight='bold')
axes[1].set_ylabel('Montant ($)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Distribution des ventes par région (carte simplifiée – barres par état)

In [ ]:
# Carte simplifiée : ventes par état (horizontal bar)
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Toutes les ventes par état
colors_state = ['#e74c3c' if v < state_sales.median() else '#2ecc71' for v in state_sales.values]
axes[0].barh(state_sales.index, state_sales.values, color=colors_state)
axes[0].set_title('Ventes par État (vert = au-dessus de la médiane)', fontweight='bold')
axes[0].set_xlabel('Ventes ($)')
axes[0].tick_params(axis='y', labelsize=8)
axes[0].axvline(x=state_sales.median(), color='blue', linestyle='--', alpha=0.5, label='Médiane')
axes[0].legend()

# Ventes par région
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
axes[1].bar(region_sales.index, region_sales.values,
            color=['#3498db', '#2ecc71', '#e67e22', '#9b59b6'], edgecolor='white')
axes[1].set_title('Ventes par Région', fontweight='bold')
axes[1].set_ylabel('Ventes ($)')
for i, val in enumerate(region_sales.values):
    axes[1].text(i, val + 2000, f'${val:,.0f}', ha='center', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Partie 3 – Visualisations avec Seaborn

### 3.1 Top 10 Produits par Ventes

In [ ]:
top10_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 7))
ax = sns.barplot(x=top10_products.values, y=top10_products.index, palette='Blues_r')

plt.title('Top 10 Produits par Ventes Totales', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Ventes ($)', fontweight='bold')
plt.ylabel('Produit', fontweight='bold')

max_val = top10_products.values.max()
for i, val in enumerate(top10_products.values):
    ax.text(val + max_val * 0.01, i, f'${val:,.0f}', va='center', fontsize=9)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Produit #1 : {top10_products.index[0]}')
print(f'Ventes     : ${top10_products.iloc[0]:,.0f}')

### 3.2 Relation entre Discount et Profit (Scatter Plot)

In [ ]:
plt.figure(figsize=(13, 7))

# Scatter par catégorie
sns.scatterplot(data=df, x='Discount', y='Profit',
                hue='Category', alpha=0.55, s=45)

# Ligne de tendance globale
sns.regplot(data=df, x='Discount', y='Profit',
            scatter=False, color='red',
            line_kws={'linewidth': 2, 'linestyle': '--'},
            label='Tendance globale')

# Ligne de seuil de rentabilité
plt.axhline(y=0, color='black', linewidth=1, alpha=0.5, linestyle='-')
plt.text(0.55, 15, 'Seuil de rentabilité', fontsize=9, alpha=0.6)

plt.title('Impact du Discount sur le Profit par Catégorie', fontsize=13, fontweight='bold')
plt.xlabel('Taux de Remise (Discount)', fontweight='bold')
plt.ylabel('Profit ($)', fontweight='bold')
plt.legend(title='Catégorie', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Statistiques par niveau de remise
bins          = [0, 0.1, 0.2, 0.3, 0.5, 1.0]
labels        = ['0-10%', '10-20%', '20-30%', '30-50%', '50-100%']
df['disc_bin'] = pd.cut(df['Discount'], bins=bins, labels=labels, include_lowest=True)
discount_prof = df.groupby('disc_bin', observed=True)['Profit'].mean().round(2)
print('Profit moyen par tranche de remise :')
print(discount_prof)

### 3.3 Ventes par catégorie et segment (heatmap)

In [ ]:
pivot_sales = df.pivot_table(
    values='Sales',
    index='Category',
    columns='Segment',
    aggfunc='sum'
).round(0)

plt.figure(figsize=(9, 4))
sns.heatmap(pivot_sales, annot=True, fmt=',.0f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'Ventes ($)'})
plt.title('Ventes par Catégorie et Segment Client', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Distribution des profits par sous-catégorie

In [ ]:
# Profit médian par sous-catégorie pour trier le boxplot
subcat_order = (df.groupby('Sub-Category')['Profit']
                  .median()
                  .sort_values(ascending=False)
                  .index.tolist())

plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='Sub-Category', y='Profit',
            order=subcat_order, palette='RdYlGn',
            showfliers=False)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title('Distribution du Profit par Sous-Catégorie', fontsize=13, fontweight='bold')
plt.xlabel('Sous-Catégorie')
plt.ylabel('Profit ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Partie 4 – Analyse Comparative : Matplotlib vs Seaborn

In [ ]:
import time

# Même graphique avec les deux outils pour comparer
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Version Matplotlib
cat_sales = df.groupby('Category')['Sales'].sum()
axes[0].bar(cat_sales.index, cat_sales.values, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Ventes par Catégorie (Matplotlib)', fontweight='bold')
axes[0].set_ylabel('Ventes ($)')
for i, val in enumerate(cat_sales.values):
    axes[0].text(i, val + 1000, f'${val:,.0f}', ha='center', fontsize=9)

# Version Seaborn
cat_df = cat_sales.reset_index()
cat_df.columns = ['Category', 'Sales']
sns.barplot(data=cat_df, x='Category', y='Sales', palette='Set2', ax=axes[1])
axes[1].set_title('Ventes par Catégorie (Seaborn)', fontweight='bold')
axes[1].set_ylabel('Ventes ($)')

plt.tight_layout()
plt.show()

# Mesure des performances
start = time.time()
plt.figure(figsize=(6, 4))
plt.bar(cat_sales.index, cat_sales.values)
plt.close()
t_mpl = time.time() - start

start = time.time()
plt.figure(figsize=(6, 4))
sns.barplot(data=cat_df, x='Category', y='Sales')
plt.close()
t_sns = time.time() - start

print(f'Temps Matplotlib : {t_mpl:.4f}s')
print(f'Temps Seaborn    : {t_sns:.4f}s')

### Observations sur Matplotlib vs Seaborn

| Critère | Matplotlib | Seaborn |
|---|---|---|
| **Facilité d'utilisation** | Moyen (plus verbeux) | Élevée (moins de code) |
| **Personnalisation** | Très haute | Moyenne |
| **Style par défaut** | Basique | Professionnel |
| **Stats intégrées** | Non | Oui (regplot, boxplot...) |
| **Interactivité** | Avec ipywidgets | Limité |
| **Vitesse** | Plus rapide | Un peu plus lent |

**Recommandation :**  
- Utiliser **Matplotlib** pour les graphiques interactifs (avec `ipywidgets`) et les visualisations très personnalisées.  
- Utiliser **Seaborn** pour l'exploration rapide des données et les présentations, grâce à ses styles attractifs et ses fonctions statistiques intégrées.

## Partie 5 – Résumé et Insights Clés

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
overall_margin = total_profit / total_sales * 100

top_state     = df.groupby('State')['Sales'].sum().idxmax()
top_category  = df.groupby('Category')['Sales'].sum().idxmax()
top_product   = df.groupby('Product Name')['Sales'].sum().idxmax()
loss_pct      = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100

print('=== INSIGHTS CLÉS – US SUPERSTORE ===')
print()
print(f'Chiffre d\'affaires total   : ${total_sales:,.0f}')
print(f'Profit total               : ${total_profit:,.0f}')
print(f'Marge bénéficiaire globale : {overall_margin:.1f}%')
print()
print(f'État le plus vendeur       : {top_state}')
print(f'Catégorie leader           : {top_category}')
print(f'Produit #1 en ventes       : {top_product}')
print()
print(f'% de ventes à perte (discount > 20%) : {loss_pct:.1f}%')
print()
print('RECOMMANDATIONS :')
print('  1. Concentrer les efforts marketing sur la Californie et New York')
print('  2. Limiter les remises à 20% maximum pour préserver les marges')
print('  3. Prioriser la catégorie Technology (meilleures marges)')
print('  4. Réduire les investissements dans les sous-catégories déficitaires (Tables, Bookcases)')